# Assignment 2 DSC 102 FA25

## Introduction

In this assignment we will conduct data engineering for the Amazon dataset. It is divided into 2 parts. The extracted features in Part 1 will be used for the Part 2 of assignment, where you train a model (or models) to predict user ratings for a product.

We will be using Apache Spark for this assignment. The default Spark API will be DataFrame, as it is now the recommended choice over the RDD API. That being said, please feel free to switch back to the RDD API if you see it as a better fit for the task. We provide you an option to request RDD format to start with. Also you can switch between DataFrame and RDD in your solution. 

Another newer API is Koalas, which is also avaliable. However, it has constraints and is not applicable to most tasks. Refer to the PA statement for detail.

### Set the following parameters

In [1]:
PID = 'A17832678' # your pid, for instance: 'a43223333'
INPUT_FORMAT = 'dataframe' # choose a format of your input data, valid options: 'dataframe', 'rdd', 'koalas'

In [2]:
# Boiler plates, do NOT modify
%load_ext autoreload
%autoreload 2
import os
import getpass
from pyspark.sql import SparkSession
from utilities import SEED
from utilities import PA2Test
from utilities import PA2Data
from utilities import data_cat
from pa2_main import PA2Executor
import time
if INPUT_FORMAT == 'dataframe':
    import pyspark.ml as M
    import pyspark.sql.functions as F
    import pyspark.sql.types as T
if INPUT_FORMAT == 'koalas':
    import databricks.koalas as ks
elif INPUT_FORMAT == 'rdd':
    import pyspark.mllib as M
    from pyspark.mllib.feature import Word2Vec
    from pyspark.mllib.linalg import Vectors
    from pyspark.mllib.linalg.distributed import RowMatrix

os.environ['PYSPARK_SUBMIT_ARGS'] = '--py-files utilities.py,assignment2.py \
--deploy-mode client \
pyspark-shell'

class args:
    review_filename = data_cat.review_filename
    product_filename = data_cat.product_filename
    product_processed_filename = data_cat.product_processed_filename
    ml_features_train_filename = data_cat.ml_features_train_filename
    ml_features_test_filename = data_cat.ml_features_test_filename
    output_root = '/home/{}/{}-pa2/test_results'.format(getpass.getuser(), PID)
    test_results_root = data_cat.test_results_root
    pid = PID

pa2 = PA2Executor(args, input_format=INPUT_FORMAT)
data_io = pa2.data_io
data_dict = pa2.data_dict
begin = time.time()


26/05/31 12:19:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/bitnami/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Loading datasets ...Done


In [3]:
# Import your own dependencies

from pyspark import StorageLevel

#-----------------------------

# Part 1: Feature Engineering

In [4]:
# Bring the part_1 datasets to memory and de-cache part_2 datasets. 
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_1')

# Task0: warm up 
This task is provided for you to get familiar with Spark API. We will use the dataframe API to demonstrate. Solution is given to you and this task won't be graded.

Refer to https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html for API guide.

The task is to implement the function below. Given the ```product_data``` table:
1. Take and print five rows.

1. Select only the ```asin``` column, then print five rows of it.

1. Select the row where ```asin = B00I8KEOTM``` and print it.

1. Count the total number of rows.

1. Calculate the mean ```price```.

1. You need to conduct the above operations, then extract some statistics out of the generated columns. You need to put the statistics in a python dictionary named ```res```. The description and schema of it are as follows:
    ```
    res
     | -- count_total: int -- count of total rows of the entire table after your operations
     | -- mean_price: float -- mean value of column price
    ```

In [5]:
def task_0(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    product_data.show(5)
    product_data[['asin']].show(5)
    product_data.where(F.col('asin') == 'B00I8KEOTM').show()
    count_rows = product_data.count()
    mean_price = product_data.select(F.avg(F.col('price'))).head()[0]
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmatically. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {'count_total': None, 'mean_price': None}
    
    # Modify res:

    res['count_total'] = count_rows
    res['mean_price'] = mean_price

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    return res
    # -------------------------------------------------------------------------

In [6]:
if INPUT_FORMAT == 'dataframe':
    res = task_0(data_io, data_dict['product'])
    pa2.tests.test(res, 'task_0')

+----------+--------------------+--------------------+--------------------+-----+--------------------+
|      asin|           salesRank|          categories|               title|price|             related|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
|B00I8HVV6E|{Home &amp; Kitch...|[[Home & Kitchen,...|Intelligent Desig...|27.99|{also_viewed -> [...|
|B00I8KEOTM|                null|[[Apps for Androi...|                null| null|{also_viewed -> [...|
|B00I8KCW4G|{Clothing -> 2233...|[[Clothing, Shoes...|eShakti Women's P...|41.95|{also_viewed -> [...|
|B00I8JKCQW|{Clothing -> 1405...|[[Clothing, Shoes...|Lady Slimming Mid...| null|{also_viewed -> [...|
|B00I8JKI8E|{Home &amp; Kitch...|[[Clothing, Shoes...|3 Tier Bangle Bra...|24.99|{also_viewed -> [...|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
only showing top 5 rows

+----------+
|      asin|
+----------+
|B00I8HVV

tests for task_0 --------------------------------------------------------------
2/2 passed
-------------------------------------------------------------------------------


# Task1

In [7]:
# %load -s task_1 assignment2.py
def task_1(data_io, review_data, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    agg_reviews = review_data.groupBy("asin").agg(
        F.avg("overall").alias('meanRating'),
        F.count("overall").alias("countRating")
    )

    joined_df = product_data.join(agg_reviews, on="asin", how="left").persist(StorageLevel.MEMORY_AND_DISK)

    stats = joined_df.select(
        F.avg("meanRating"), 
        F.variance("meanRating"), 
        F.sum(F.col("meanRating").isNull().cast("int")),
        F.avg("countRating"), 
        F.variance("countRating"), 
        F.sum(F.col("countRating").isNull().cast("int"))
    ).collect()[0]

    count_total = joined_df.count()

    joined_df.unpersist()

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmaticly. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {
        'count_total': count_total,
        'mean_meanRating': stats[0],
        'variance_meanRating': stats[1],
        'numNulls_meanRating': stats[2],
        'mean_countRating': stats[3],
        'variance_countRating': stats[4],
        'numNulls_countRating': stats[5]
    }
    # Modify res:
    



    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_1')
    return res
    # -------------------------------------------------------------------------


In [8]:
res = task_1(data_io, data_dict['review'], data_dict['product'])
pa2.tests.test(res, 'task_1')

tests for task_1 --------------------------------------------------------------
Test 1/7 : count_total ... Pass
Test 2/7 : mean_countRating ... Pass
Test 3/7 : mean_meanRating ... Pass
Test 4/7 : numNulls_countRating ... Pass
Test 5/7 : numNulls_meanRating ... Pass
Test 6/7 : variance_countRating ... Pass
Test 7/7 : variance_meanRating ... Pass
7/7 passed
-------------------------------------------------------------------------------


True


# Task 2

In [9]:
# %load -s task_2 assignment2.py
def task_2(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    salesRank_column = 'salesRank'
    categories_column = 'categories'
    asin_column = 'asin'
    # Outputs:
    category_column = 'category'
    bestSalesCategory_column = 'bestSalesCategory'
    bestSalesRank_column = 'bestSalesRank'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    
    df_with_cat = product_data.withColumn(
        "category", 
        F.when(F.col("categories")[0][0] != "", F.col("categories")[0][0]).otherwise(None)
    )
    
    df_with_sales = df_with_cat.withColumn("map_keys", F.map_keys(F.col("salesRank")))\
                               .withColumn("map_vals", F.map_values(F.col("salesRank")))
    
    processed_df = df_with_sales.withColumn(
        "bestSalesCategory", F.col("map_keys")[0]
    ).withColumn(
        "bestSalesRank", F.col("map_vals")[0]
    ).persist(StorageLevel.MEMORY_AND_DISK)
    
    count_total = processed_df.count()
    stats = processed_df.select(
        F.avg("bestSalesRank"),
        F.variance("bestSalesRank"),
        F.sum(F.col("category").isNull().cast("int")),
        F.countDistinct("category"),
        F.sum(F.col("bestSalesCategory").isNull().cast("int")),
        F.countDistinct("bestSalesCategory")
    ).collect()[0]
    
    processed_df.unpersist()
    
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'mean_bestSalesRank': stats[0],
        'variance_bestSalesRank': stats[1],
        'numNulls_category': stats[2],
        'countDistinct_category': stats[3],
        'numNulls_bestSalesCategory': stats[4],
        'countDistinct_bestSalesCategory': stats[5]
    }
    # Modify res:




    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_2')
    return res
    # -------------------------------------------------------------------------


In [10]:
res = task_2(data_io, data_dict['product'])
pa2.tests.test(res, 'task_2')

tests for task_2 --------------------------------------------------------------
Test 1/7 : countDistinct_bestSalesCategory ... Pass
Test 2/7 : countDistinct_category ... Pass
Test 3/7 : count_total ... Pass
Test 4/7 : mean_bestSalesRank ... Pass
Test 5/7 : numNulls_bestSalesCategory ... Pass
Test 6/7 : numNulls_category ... Pass
Test 7/7 : variance_bestSalesRank ... Pass
7/7 passed
-------------------------------------------------------------------------------


True

# Task 3





In [11]:
# %load -s task_3 assignment2.py
def task_3(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    price_column = 'price'
    attribute = 'also_viewed'
    related_column = 'related'
    # Outputs:
    meanPriceAlsoViewed_column = 'meanPriceAlsoViewed'
    countAlsoViewed_column = 'countAlsoViewed'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    
    df_with_count = product_data.withColumn(
        "countAlsoViewed",
        F.when(
            F.col("related.also_viewed").isNotNull() & (F.size(F.col("related.also_viewed")) > 0),
            F.size(F.col("related.also_viewed"))
        ).otherwise(None)
    )
    
    # Explode array to cross-reference product pricing
    exploded_df = product_data.select("asin", F.explode("related.also_viewed").alias("viewed_asin"))
    prices_lookup = product_data.select(F.col("asin").alias("viewed_asin"), F.col("price").alias("viewed_price"))
    
    joined_prices = exploded_df.join(prices_lookup, on="viewed_asin", how="inner")\
                               .filter(F.col("viewed_price").isNotNull())
    
    mean_prices_df = joined_prices.groupBy("asin").agg(F.avg("viewed_price").alias("meanPriceAlsoViewed"))
    
    final_df = df_with_count.join(mean_prices_df, on="asin", how="left").persist(StorageLevel.MEMORY_AND_DISK)
    
    count_total = final_df.count()
    stats = final_df.select(
        F.avg("meanPriceAlsoViewed"),
        F.variance("meanPriceAlsoViewed"),
        F.sum(F.col("meanPriceAlsoViewed").isNull().cast("int")),
        F.avg("countAlsoViewed"),
        F.variance("countAlsoViewed"),
        F.sum(F.col("countAlsoViewed").isNull().cast("int"))
    ).collect()[0]
    
    final_df.unpersist()

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'mean_meanPriceAlsoViewed': stats[0],
        'variance_meanPriceAlsoViewed': stats[1],
        'numNulls_meanPriceAlsoViewed': stats[2],
        'mean_countAlsoViewed': stats[3],
        'variance_countAlsoViewed': stats[4],
        'numNulls_countAlsoViewed': stats[5]
    }
    # Modify res:


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_3')
    return res
    # -------------------------------------------------------------------------


In [12]:
res = task_3(data_io, data_dict['product'])
pa2.tests.test(res, 'task_3')

tests for task_3 --------------------------------------------------------------
Test 1/7 : count_total ... Pass
Test 2/7 : mean_countAlsoViewed ... Pass
Test 3/7 : mean_meanPriceAlsoViewed ... Pass
Test 4/7 : numNulls_countAlsoViewed ... Pass
Test 5/7 : numNulls_meanPriceAlsoViewed ... Pass
Test 6/7 : variance_countAlsoViewed ... Pass
Test 7/7 : variance_meanPriceAlsoViewed ... Pass
7/7 passed
-------------------------------------------------------------------------------


True

# Task 4

In [13]:
# %load -s task_4 assignment2.py
def task_4(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    price_column = 'price'
    title_column = 'title'
    # Outputs:
    meanImputedPrice_column = 'meanImputedPrice'
    medianImputedPrice_column = 'medianImputedPrice'
    unknownImputedTitle_column = 'unknownImputedTitle'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    df = product_data.withColumn("price", F.col("price").cast("float"))
    
    mean_val = df.select(F.avg(F.when(~F.isnan(F.col("price")), F.col("price")))).collect()[0][0]
    mean_price = float(mean_val) if mean_val is not None else 0.0
    
    quantiles = df.filter(~F.isnan(F.col("price")) & F.col("price").isNotNull()).approxQuantile("price", [0.5], 0.001)
    median_price = float(quantiles[0]) if (quantiles and quantiles[0] is not None) else 0.0
    
    imputed_df = df.withColumn(
        "meanImputedPrice",
        F.coalesce(F.when(F.col("price").isNotNull() & F.isnan(F.col("price")), None).otherwise(F.col("price")), F.lit(mean_price))
    ).withColumn(
        "medianImputedPrice",
        F.coalesce(F.when(F.col("price").isNotNull() & F.isnan(F.col("price")), None).otherwise(F.col("price")), F.lit(median_price))
    ).withColumn(
        "unknownImputedTitle",
        F.when((F.col("title").isNull()) | (F.col("title") == ""), "unknown").otherwise(F.col("title"))
    ).persist(StorageLevel.MEMORY_AND_DISK)
                   
    count_total = imputed_df.count()
    stats = imputed_df.select(
        F.avg("meanImputedPrice"),
        F.variance("meanImputedPrice"),
        F.sum(F.col("meanImputedPrice").isNull().cast("int")),
        F.avg("medianImputedPrice"),
        F.variance("medianImputedPrice"),
        F.sum(F.col("medianImputedPrice").isNull().cast("int")),
        F.sum((F.col("unknownImputedTitle") == "unknown").cast("int"))
    ).collect()[0]
    
    imputed_df.unpersist()

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'mean_meanImputedPrice': stats[0],
        'variance_meanImputedPrice': stats[1],
        'numNulls_meanImputedPrice': stats[2],
        'mean_medianImputedPrice': stats[3],
        'variance_medianImputedPrice': stats[4],
        'numNulls_medianImputedPrice': stats[5],
        'numUnknowns_unknownImputedTitle': stats[6]
    }
    # Modify res:



    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_4')
    return res
    # -------------------------------------------------------------------------


In [14]:
res = task_4(data_io, data_dict['product'])
pa2.tests.test(res, 'task_4')

tests for task_4 --------------------------------------------------------------
Test 1/8 : count_total ... Pass
Test 2/8 : mean_meanImputedPrice ... Pass
Test 3/8 : mean_medianImputedPrice ... Pass
Test 4/8 : numNulls_meanImputedPrice ... Pass
Test 5/8 : numNulls_medianImputedPrice ... Pass
Test 6/8 : numUnknowns_unknownImputedTitle ... Pass
Test 7/8 : variance_meanImputedPrice ... Pass
Test 8/8 : variance_medianImputedPrice ... Pass
8/8 passed
-------------------------------------------------------------------------------


True

# Task 5

In [15]:
# %load -s task_5 assignment2.py
def task_5(data_io, product_processed_data, word_0, word_1, word_2):
    # -----------------------------Column names--------------------------------
    # Inputs:
    title_column = 'title'
    # Outputs:
    titleArray_column = 'titleArray'
    titleVector_column = 'titleVector'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    
    tokenized_df = product_processed_data.withColumn(
        "titleArray", 
        F.split(F.lower(F.col("title")), " ")
    )
    
    word2Vec = (M.feature.Word2Vec()
                .setVectorSize(16)
                .setMinCount(100)
                .setSeed(SEED)
                .setNumPartitions(4)
                .setInputCol(titleArray_column)
                .setOutputCol(titleVector_column))
    
    model = word2Vec.fit(tokenized_df)
    transformed_df = model.transform(tokenized_df)


    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': transformed_df.count(),
        'size_vocabulary': model.getVectors().count(),
        'word_0_synonyms': [(row[0], float(row[1])) for row in model.findSynonyms(word_0, 10).collect()],
        'word_1_synonyms': [(row[0], float(row[1])) for row in model.findSynonyms(word_1, 10).collect()],
        'word_2_synonyms': [(row[0], float(row[1])) for row in model.findSynonyms(word_2, 10).collect()]
    }
    # Modify res:
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_5')
    return res
    # -------------------------------------------------------------------------


In [16]:
res = task_5(data_io, data_dict['product_processed'], 'piano', 'rice', 'laptop')
pa2.tests.test(res, 'task_5')

26/05/31 12:47:02 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 157386 ms exceeds timeout 120000 ms
26/05/31 12:47:02 ERROR TaskSchedulerImpl: Lost executor 0 on 10.32.73.136: Executor heartbeat timed out after 157386 ms
26/05/31 12:47:02 WARN TaskSetManager: Lost task 0.0 in stage 103.0 (TID 2066) (10.32.73.136 executor 0): ExecutorLostFailure (executor 0 exited caused by one of the running tasks) Reason: Executor heartbeat timed out after 157386 ms
26/05/31 12:47:02 WARN TaskSetManager: Lost task 3.0 in stage 103.0 (TID 2069) (10.32.73.136 executor 0): ExecutorLostFailure (executor 0 exited caused by one of the running tasks) Reason: Executor heartbeat timed out after 157386 ms
26/05/31 12:47:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_64_10 !
26/05/31 12:47:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_88_8 !
26/05/31 12:47:02 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_64_3 !
26/05/

26/05/31 12:48:47 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/31 12:48:47 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS


tests for task_5 --------------------------------------------------------------
Test 1/8 : count_total ... Pass
Test 2/8 : size_vocabulary ... Pass
Test 3/8 : word_0_synonyms-length ... Pass
Test 4/8 : word_0_synonyms-correctness ... Pass
Test 5/8 : word_1_synonyms-length ... Pass
Test 6/8 : word_1_synonyms-correctness ... Pass
Test 7/8 : word_2_synonyms-length ... Pass
Test 8/8 : word_2_synonyms-correctness ... Pass
5/5 passed
-------------------------------------------------------------------------------


True

# Task 6

In [ ]:
# %load -s task_6 assignment2.py
def task_6(data_io, product_processed_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    category_column = 'category'
    # Outputs:
    categoryIndex_column = 'categoryIndex'
    categoryOneHot_column = 'categoryOneHot'
    categoryPCA_column = 'categoryPCA'
    # -------------------------------------------------------------------------    

    # ---------------------- Your implementation begins------------------------

    indexer = M.feature.StringIndexer(inputCol="category", outputCol="categoryIndex")
    indexed_df = indexer.fit(product_processed_data).transform(product_processed_data)
    
    encoder = M.feature.OneHotEncoder(inputCol="categoryIndex", outputCol="categoryOneHot", dropLast=False)
    encoded_df = encoder.fit(indexed_df).transform(indexed_df)
    
    pca = M.feature.PCA(k=15, inputCol="categoryOneHot", outputCol="categoryPCA")
    pca_model = pca.fit(encoded_df)
    transformed_df = pca_model.transform(encoded_df).cache()
    
    summary_row = transformed_df.select(
        M.stat.Summarizer.mean(F.col("categoryOneHot")),
        M.stat.Summarizer.mean(F.col("categoryPCA"))
    ).collect()[0]

    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': int(transformed_df.count()),
        'meanVector_categoryOneHot': [float(x) for x in summary_row[0].toArray()],
        'meanVector_categoryPCA': [float(x) for x in summary_row[1].toArray()]
    }
    # Modify res:




    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_6')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_6(data_io, data_dict['product_processed'])
pa2.tests.test(res, 'task_6')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))

# Part 2: Model Selection

In [ ]:
# Bring the part_2 datasets to memory and de-cache part_1 datasets.
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_2')

# Task 7

In [ ]:
def task_7(data_io, train_data, test_data):
    
    # ---------------------- Your implementation begins------------------------
    
    
    
    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': None
    }
    # Modify res:


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_7')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_7(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_7')

# Task 8

In [ ]:
def task_8(data_io, train_data, test_data):
    
    # ---------------------- Your implementation begins------------------------
    
    
    
    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': None,
        'valid_rmse_depth_5': None,
        'valid_rmse_depth_7': None,
        'valid_rmse_depth_9': None,
        'valid_rmse_depth_12': None,
    }
    # Modify res:


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_8')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_8(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa2.tests.test(res, 'task_8')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))